# Chapter 8 - From Models to Systems

Companion notebook for *Video Generation with AI* (O'Reilly) by Joseph Enochs.

**Learning objective:** understand what changes when a generative AI model becomes a product, by implementing the eight architectural patterns that separate production AI applications from research scripts and building the same custom extension for LTX Desktop, ComfyUI, and InvokeAI.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenochs/video-ai-book-companion/blob/main/chapter-08/notebooks/chapter_8_companion_notebook.ipynb)

This is a systems chapter, so the notebook works differently from earlier chapters: no GPU is required. Every pattern runs as an in-notebook demonstration, with service and API code invoked directly as handler functions rather than started as long-lived servers. Cells that belong to a specific host application (Electron, ComfyUI, InvokeAI) are shown as clearly labeled reference listings or written to files for installation, never executed here.

In [ ]:
# Install the one third-party dependency used for the typed-boundary demo.
# Everything else in this chapter is Python standard library.
%pip install -q "pydantic>=2.6"


In [ ]:
import sys

assert sys.version_info >= (3, 10), "This notebook uses structural pattern matching (Python 3.10+)"

import hmac
import os
import platform
import secrets
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime
from functools import wraps
from pathlib import Path
from typing import Callable, Protocol, Union
from uuid import UUID, uuid4

print(sys.version)


## The gap between a model and a product

By Chapter 7 you could load LTX-2's checkpoint and drive it from Python. A model is not a product: production brings license state, capability detection, downloads with recovery, progress streaming, updates, telemetry, and security. This chapter studies how three reference architectures answer that work: LTX Desktop (Electron + React + Python, tight typing), ComfyUI (community plugin mesh, loose contracts), and InvokeAI (service-oriented monolith, typed invocations).

## Pattern 1: The two-process architecture

One process paints pixels (the Electron renderer), one process runs inference (the Python backend), and the boundary between them is explicit, typed, and observable. There is no Python listing for this pattern; the orchestration lives in `electron/python-backend.ts`, which spawns, monitors, and restarts the backend, adopts stale processes still bound to the port, and publishes `{status: 'restarting'}` or `{status: 'dead', exitCode}` events the renderer subscribes to.

The discipline to remember: the renderer talks to Python only over HTTP or WebSocket, never over OS primitives.

## Pattern 2: Backend as source of truth

Capability decisions (local vs hosted API, GPU sufficiency) are computed once at Python startup, stored once, and read by everything. The decision function from `backend/runtime_config/runtime_policy.py` is pure and testable.

In [ ]:
# From backend/runtime_config/runtime_policy.py (LTX Desktop @ 81f7f45)
def decide_force_api_generations(
    system: str,
    cuda_available: bool,
    vram_gb: int | None,
) -> bool:
    if system == "Darwin":          # Always API-only on macOS
        return True
    if system in ("Windows", "Linux"):
        if not cuda_available:
            return True
        if vram_gb is None:
            return True
        return vram_gb < 15         # <15GB VRAM forces API mode
    return True                      # Fail closed for unknown platforms


In [ ]:
# Exercise the decision tree across representative machines.
scenarios = [
    ("Darwin",  False, None),  # macOS laptop
    ("Windows", False, None),  # Windows, no CUDA
    ("Linux",   True,  8),     # Linux, small GPU
    ("Linux",   True,  24),    # Linux, RTX 4090 class
    ("Linux",   True,  None),  # CUDA present, VRAM unknown
    ("FreeBSD", True,  24),    # unknown platform: fail closed
]
for system, cuda, vram in scenarios:
    forced = decide_force_api_generations(system, cuda, vram)
    mode = "API" if forced else "LOCAL"
    print(f"{system:8s} cuda={cuda!s:5s} vram={str(vram):4s} -> {mode}")


In [ ]:
# The decision is published through one endpoint and read once at startup:
#   GET /api/runtime-policy -> { "force_api_generations": bool }
# Here we invoke the handler logic directly instead of running a server.
def get_runtime_policy_for_this_machine() -> dict:
    system = platform.system()
    cuda_available, vram_gb = False, None
    try:
        import torch
        cuda_available = torch.cuda.is_available()
        if cuda_available:
            vram_gb = int(torch.cuda.get_device_properties(0).total_memory / 1024**3)
    except Exception:
        pass
    return {"force_api_generations": decide_force_api_generations(system, cuda_available, vram_gb)}

print(get_runtime_policy_for_this_machine())


## Pattern 3: The typed IPC bridge

Every IPC method's input and output is declared once in a shared schema; both sides derive from it, so the contract cannot drift silently. The TypeScript listings below are reference only (they run inside Electron, not here).

```ts
// From electron/preload.ts (LTX Desktop @ 81f7f45)
import { contextBridge } from 'electron';
import { electronAPISchemas } from '../shared/electron-api-schema';

const electronAPI = buildClientFromSchemas(electronAPISchemas);
contextBridge.exposeInMainWorld('electronAPI', electronAPI);
```

```ts
// From shared/electron-api-schema.ts (LTX Desktop @ 81f7f45)
export const electronAPISchemas = {
  acceptLicense: {
    input:  z.void(),
    output: z.object({ accepted: z.boolean() }),
  },
  fetchLicenseText: {
    input:  z.void(),
    output: z.object({ text: z.string() }),
  },
  approvePath: {
    input:  z.object({ path: z.string() }),
    output: z.object({ approved: z.boolean() }),
  },
  // ... all other IPC methods
} as const;
```

```ts
// From electron/ipc/typed-handle.ts (LTX Desktop @ 81f7f45)
export function handle<K extends keyof Schemas>(
  key: K,
  handler: (input: z.infer<Schemas[K]['input']>)
           => Promise<z.infer<Schemas[K]['output']>>
) {
  ipcMain.handle(key, async (_event, raw) => {
    const input = Schemas[key].input.parse(raw);    // validate at boundary
    const output = await handler(input);
    return Schemas[key].output.parse(output);        // validate response shape
  });
}
```

The same validate-at-the-boundary idea translates directly to Python with Pydantic, which is what the Python backend uses for its HTTP contract. The runnable analog:

In [ ]:
# Python analog of the typed bridge: one schema object, both sides derive from it.
from pydantic import BaseModel


class ApprovePathInput(BaseModel):
    path: str


class ApprovePathOutput(BaseModel):
    approved: bool


SCHEMAS = {"approvePath": (ApprovePathInput, ApprovePathOutput)}
HANDLERS = {}


def handle(key, handler):
    input_model, output_model = SCHEMAS[key]

    def wrapper(raw: dict) -> dict:
        inp = input_model.model_validate(raw)        # validate at boundary
        out = handler(inp)
        return output_model.model_validate(out).model_dump()  # validate response shape

    HANDLERS[key] = wrapper


handle("approvePath", lambda inp: {"approved": inp.path.startswith("/outputs")})

print(HANDLERS["approvePath"]({"path": "/outputs/video.mp4"}))
try:
    HANDLERS["approvePath"]({"path": 42})  # malformed payload rejected at the boundary
except Exception as e:
    print(type(e).__name__, "raised for malformed input")


## Pattern 4: Localhost as a security boundary

"It is just localhost" ages badly: any process on the machine can reach the port. LTX Desktop authenticates every request with per-session tokens, compares them in constant time, enforces CSP from the main process, and validates every user-controllable path.

In [ ]:
# Per-session tokens. The Electron original (electron/python-backend.ts) is:
#   const LTX_AUTH_TOKEN  = randomBytes(32).toString('base64url');
#   const LTX_ADMIN_TOKEN = randomBytes(32).toString('base64url');
# Both are passed to the Python subprocess via env vars. Python equivalent:
LTX_AUTH_TOKEN = secrets.token_urlsafe(32)
LTX_ADMIN_TOKEN = secrets.token_urlsafe(32)
print("auth token: ", LTX_AUTH_TOKEN[:12] + "...")
print("admin token:", LTX_ADMIN_TOKEN[:12] + "...")


# From backend/app_factory.py (LTX Desktop @ 81f7f45)
def verify_token(provided: str, expected: str) -> bool:
    return hmac.compare_digest(provided.encode(), expected.encode())


print(verify_token(LTX_AUTH_TOKEN, LTX_AUTH_TOKEN))   # True
print(verify_token("guess", LTX_AUTH_TOKEN))          # False, in constant time


The renderer's Content Security Policy is enforced by Electron itself, so the renderer cannot relax it. Reference listing, not executed here:

```ts
// From electron/csp.ts (LTX Desktop @ 81f7f45) - production
session.defaultSession.webRequest.onHeadersReceived((details, callback) => {
  callback({
    responseHeaders: {
      ...details.responseHeaders,
      'Content-Security-Policy': [
        "default-src 'self'; "
        + "script-src 'self'; "          // no inline scripts
        + "style-src 'self' 'unsafe-inline'; "
        + "object-src 'none'; "
        + "connect-src 'self' http://127.0.0.1:* ws://127.0.0.1:*"
      ],
    },
  });
});
```

Path validation is a Python port of `electron/path-validation.ts`: whitelist roots, resolve before checking, fold case only on Windows.

In [ ]:
# Python port of electron/path-validation.ts (LTX Desktop @ 81f7f45)
from urllib.parse import unquote, urlparse

IS_WINDOWS = sys.platform == "win32"


def strip_file_url(p: str) -> str:
    return unquote(urlparse(p).path)


def normalize(p: str) -> str:
    resolved = os.path.abspath(p)
    return resolved.lower() if IS_WINDOWS else resolved


def validate_path(input_path: str, allowed_roots: list[str]) -> str:
    cleaned = strip_file_url(input_path) if input_path.startswith("file://") else input_path
    resolved = os.path.abspath(cleaned)
    norm = normalize(resolved)

    for root in (normalize(r) for r in allowed_roots):
        if norm == root or norm.startswith(root + os.sep):
            return resolved
    # per-call approvePath() additions would be checked similarly
    raise ValueError(f"Path not allowed: {input_path}")


allowed = ["/tmp/ltx-outputs"]
print(validate_path("/tmp/ltx-outputs/video.mp4", allowed))
print(validate_path("file:///tmp/ltx-outputs/take2.mp4", allowed))
for attack in ["/tmp/ltx-outputs/../../etc/passwd", "/etc/passwd"]:
    try:
        validate_path(attack, allowed)
    except ValueError as e:
        print("rejected:", e)


## Pattern 5: Model lifecycle: download, license, storage

Models are fetched after install (LTX-2.3's bf16 checkpoint alone is around 44 GB), stored under platform-canonical roots (`%LOCALAPPDATA%`, `~/Library/Application Support`, `$XDG_DATA_HOME`), with license acceptance recorded as durable state and per-file progress so three simultaneous downloads get three independent bars.

In [ ]:
# From backend/state/app_state_types.py (LTX Desktop @ 81f7f45)
ModelFileType = str  # an enum in the real codebase


@dataclass(frozen=True)
class DownloadingSession:
    session_id: UUID
    requested_files: tuple[ModelFileType, ...]
    completed_files: frozenset[ModelFileType]
    current_file: ModelFileType | None
    bytes_downloaded: int
    bytes_total: int
    download_speed_bps: float
    started_at: datetime


In [ ]:
# Every requested file is in exactly one of three categories at any moment.
session = DownloadingSession(
    session_id=uuid4(),
    requested_files=("checkpoint", "upsampler", "gemma_encoder"),
    completed_files=frozenset({"upsampler"}),
    current_file="checkpoint",
    bytes_downloaded=17_500_000_000,
    bytes_total=44_000_000_000,
    download_speed_bps=52e6,
    started_at=datetime.utcnow(),
)

for f in session.requested_files:
    if f in session.completed_files:
        status = "done"
    elif f == session.current_file:
        pct = 100 * session.bytes_downloaded / session.bytes_total
        status = f"in progress ({pct:.0f}%, {session.download_speed_bps/1e6:.0f} MB/s)"
    else:
        status = "pending"
    print(f"{f:15s} {status}")


## Pattern 6: State as discriminated unions

One nullable-field struct for every phase accumulates correctness debt; discriminated unions make illegal states unrepresentable. First the anti-pattern, then the real representation.

In [ ]:
# anti-pattern: what LTX Desktop deliberately avoids
from typing import Literal


@dataclass
class MonolithicGenerationState:
    job_id: UUID
    status: Literal["pending", "running", "complete", "error", "cancelled"]
    progress: float | None
    current_step: int | None
    output_path: str | None
    error_message: str | None
    cancelled_at: datetime | None
    completed_at: datetime | None

# Every consumer must remember which fields are valid in which status.
# A "completed" value with output_path=None is representable, so eventually it happens.


In [ ]:
# From backend/state/app_state_types.py (LTX Desktop @ 81f7f45)
@dataclass(frozen=True)
class GenerationPending:
    job_id: UUID
    submitted_at: datetime


@dataclass(frozen=True)
class GenerationRunning:
    job_id: UUID
    started_at: datetime
    progress: float                  # always present, 0.0 - 1.0
    current_step: int                # always present
    estimated_completion: datetime | None


# The manuscript defines the remaining variants "similarly"; here they are in full.
@dataclass(frozen=True)
class GenerationComplete:
    job_id: UUID
    completed_at: datetime
    output_path: str
    duration_seconds: float


@dataclass(frozen=True)
class GenerationError:
    job_id: UUID
    failed_at: datetime
    error_message: str
    error_kind: str


@dataclass(frozen=True)
class GenerationCancelled:
    job_id: UUID
    cancelled_at: datetime
    cancelled_at_step: int


GenerationState = Union[
    GenerationPending,
    GenerationRunning,
    GenerationComplete,
    GenerationError,
    GenerationCancelled,
]


In [ ]:
# Handlers consume the union with exhaustive structural pattern matching.
def describe(state: GenerationState) -> dict:
    match state:
        case GenerationPending(job_id=jid):
            return {"status": "queued", "job": str(jid)}
        case GenerationRunning(progress=p, current_step=step):
            return {"status": "running", "progress": p, "step": step}
        case GenerationComplete(output_path=path, duration_seconds=dur):
            return {"status": "done", "output": path, "elapsed_s": dur}
        case GenerationError(error_message=msg, error_kind=kind):
            return {"status": "error", "kind": kind, "message": msg}
        case GenerationCancelled(cancelled_at_step=step):
            return {"status": "cancelled", "at_step": step}


jid = uuid4()
now = datetime.utcnow()
for s in [
    GenerationPending(jid, now),
    GenerationRunning(jid, now, 0.42, 17, None),
    GenerationComplete(jid, now, "/outputs/final.mp4", 5.0),
    GenerationError(jid, now, "CUDA out of memory", "OutOfMemoryError"),
    GenerationCancelled(jid, now, 23),
]:
    print(describe(s))


## Pattern 7: Service Protocols and fakes (no mocks)

Each side effect hides behind a `typing.Protocol`. Production wires in real implementations; tests wire in hand-written fakes that satisfy the same contract, so tests describe behavior, not implementation.

In [ ]:
# From backend/services/interfaces.py (LTX Desktop @ 81f7f45)
class FastVideoPipeline(Protocol):
    def __call__(
        self,
        prompt: str,
        image: object | None,          # PIL.Image.Image in the real codebase
        size: tuple[int, int],
        duration_seconds: float,
        on_progress: Callable[[float, int], None],
    ) -> Path: ...


class ModelDownloader(Protocol):
    def download(
        self,
        file_type: ModelFileType,
        target_path: Path,
        on_progress: Callable[[int, int, float], None],
    ) -> None: ...


class GpuInfo(Protocol):
    @property
    def cuda_available(self) -> bool: ...
    @property
    def vram_gb(self) -> int | None: ...


In [ ]:
# From backend/tests/fakes.py (LTX Desktop @ 81f7f45), lightly adapted:
# the fixture file is generated here so the notebook is self-contained.
FIXTURE_TINY_MP4 = Path("fixture_tiny.mp4")
FIXTURE_TINY_MP4.write_bytes(b"\x00\x00\x00\x18ftypmp42")  # placeholder bytes


@dataclass(frozen=True)
class FakeCall:
    prompt: str
    size: tuple[int, int]
    duration: float


class FakeFastVideoPipeline:
    """Test double: returns a tiny canned video file after a short delay."""

    def __init__(self, fixture_video_path: Path = FIXTURE_TINY_MP4):
        self.fixture = fixture_video_path
        self.calls: list[FakeCall] = []   # tests can assert on this

    def __call__(self, prompt, image, size, duration_seconds, on_progress) -> Path:
        self.calls.append(FakeCall(prompt=prompt, size=size, duration=duration_seconds))
        for step in range(0, 100, 10):
            on_progress(step / 100.0, step)
            time.sleep(0.01)
        on_progress(1.0, 100)
        return self.fixture


@dataclass(frozen=True)
class FakeGpuInfo:
    cuda_available: bool = True
    vram_gb: int | None = 24


class FakeModelDownloader:
    def __init__(self):
        self.downloads: list[str] = []

    def download(self, file_type, target_path, on_progress) -> None:
        self.downloads.append(file_type)
        for i in range(1, 11):
            on_progress(i * 100_000, 1_000_000, 50e6)
        Path(target_path).write_bytes(b"\x00" * 16)


In [ ]:
# From backend/services/bundle.py (LTX Desktop @ 81f7f45), trimmed to the
# three services this notebook exercises (the real bundle holds about 12).
@dataclass
class ServiceBundle:
    fast_video_pipeline: FastVideoPipeline
    model_downloader: ModelDownloader
    gpu_info: GpuInfo


# Production wiring (reference only; needs the checkpoint and a large GPU):
#   bundle = ServiceBundle(
#       fast_video_pipeline=LTXFastVideoPipeline(checkpoint_path, gemma_root, device),
#       model_downloader=HuggingFaceModelDownloader(api_token),
#       gpu_info=GpuInfoImpl(),
#   )

# Test and notebook wiring: same seam, fakes plugged in.
bundle = ServiceBundle(
    fast_video_pipeline=FakeFastVideoPipeline(),
    model_downloader=FakeModelDownloader(),
    gpu_info=FakeGpuInfo(cuda_available=True, vram_gb=24),
)

out = bundle.fast_video_pipeline(
    prompt="a red fox at dawn", image=None, size=(768, 512),
    duration_seconds=5.0, on_progress=lambda p, s: None,
)
print("output:", out, "| recorded calls:", bundle.fast_video_pipeline.calls)


## Pattern 8: Lock-aware handlers

State mutation goes through an explicit RLock via a decorator; heavy work is submitted to a TaskRunner and never holds the lock. One doorway into the kitchen, and the cook leaves fast.

In [ ]:
# From backend/handlers/base.py (LTX Desktop @ 81f7f45)
class StateProvider:
    def __init__(self):
        self.current_generation: GenerationState | None = None


class HandlerBase:
    def __init__(self, state_provider: StateProvider):
        self._state = state_provider
        self._lock = threading.RLock()


def with_state_lock(method):
    """Acquire the handler's RLock for the duration of the method."""
    @wraps(method)
    def wrapper(self, *args, **kwargs):
        with self._lock:
            return method(self, *args, **kwargs)
    return wrapper


class TaskRunner:
    """Thin wrapper around a thread pool: lifecycle, exception capture, IDs."""

    def __init__(self, max_workers: int = 2):
        self._pool = ThreadPoolExecutor(max_workers=max_workers)
        self._futures = []

    def submit(self, fn, *args):
        self._futures.append(self._pool.submit(fn, *args))

    def wait_all(self, timeout=None):
        for f in self._futures:
            f.result(timeout=timeout)   # re-raises exceptions from workers


class GenerationAlreadyRunningError(RuntimeError):
    pass


## The landscape: three architectures, one problem

ComfyUI runs as one Python process where a custom node is a plain class dropped into `custom_nodes/`; InvokeAI is a service-oriented monolith with typed Pydantic invocations, a session queue, and around 30 injected services. The core ComfyUI contract, for reference (runs inside ComfyUI, not here):

```python
# From nodes.py (ComfyUI @ 5e3f15a)
class CLIPTextEncode(ComfyNodeABC):
    @classmethod
    def INPUT_TYPES(s) -> InputTypeDict:
        return {
            "required": {
                "text": (IO.STRING, {"multiline": True, ...}),
                "clip": (IO.CLIP, {"tooltip": "..."})
            }
        }
    RETURN_TYPES = (IO.CONDITIONING,)
    FUNCTION = "encode"
    CATEGORY = "conditioning"

    def encode(self, clip, text):
        tokens = clip.tokenize(text)
        return (clip.encode_from_tokens_scheduled(tokens),)
```

A condensed view of Table 8-1:

| Concern | LTX Desktop | ComfyUI | InvokeAI |
|---|---|---|---|
| Process model | Electron + Python | Single Python process | Single Python process |
| Extension contract | Vendor-controlled, no SDK | Duck-typed class + dict registry | Typed Pydantic invocation |
| Type safety | Pydantic + Zod, strict | Mostly duck-typed | Strict Pydantic |
| Auth on local API | Per-session bearer tokens | None by default | None by default |
| Primary tradeoff | Rigor, narrower extension | Breadth, weaker contracts | Rigor, mid-sized ecosystem |

## Capstone: the same feature on three platforms

We expose LTX-2.3's audio-to-video pipeline (`a2vid_two_stage` from Chapter 7) as a user-accessible feature in each architecture. The pipeline call is identical everywhere; the scaffolding differs, and the line counts (roughly 280, 50, and 80) tell the architectural story.

### LTX Desktop: a cross-layer code change

No public extension API, so the feature ships as a code change: a handler in the Python backend, a thin FastAPI route, and one typed fetch in the renderer. The handler below is the manuscript's `AudioToVideoHandler`, runnable here because the pipeline arrives through the ServiceBundle seam: we inject the fake pipeline where production injects `A2VidPipelineTwoStage`.

In [ ]:
# From backend/handlers/audio_to_video_handler.py (companion @ ch8-capstone),
# with the ServiceBundle pipeline standing in for A2VidPipelineTwoStage so the
# handler runs end to end without a GPU.
@dataclass(frozen=True)
class A2VidRequest:
    audio_path: str
    prompt: str = ""
    duration_s: float = 5.0
    size: tuple[int, int] = (768, 512)


class AudioToVideoHandler(HandlerBase):
    def __init__(self, services: ServiceBundle, state_provider: StateProvider,
                 task_runner: TaskRunner):
        super().__init__(state_provider)
        self._services = services
        self._task_runner = task_runner

    @with_state_lock
    def start(self, request: A2VidRequest) -> UUID:
        # Holds the lock briefly for state mutation only.
        if isinstance(self._state.current_generation, (GenerationPending, GenerationRunning)):
            raise GenerationAlreadyRunningError()
        job_id = uuid4()
        self._state.current_generation = GenerationPending(
            job_id=job_id,
            submitted_at=datetime.utcnow(),
        )
        # Lock releases when this method returns.
        self._task_runner.submit(self._run, job_id, request)
        return job_id

    @with_state_lock
    def snapshot(self) -> GenerationState | None:
        return self._state.current_generation

    def _run(self, job_id: UUID, request: A2VidRequest) -> None:
        # Runs in a background thread. Does NOT hold the lock.
        started = datetime.utcnow()

        def on_progress(progress: float, step: int) -> None:
            with self._lock:
                self._state.current_generation = GenerationRunning(
                    job_id=job_id, started_at=started,
                    progress=progress, current_step=step,
                    estimated_completion=None,
                )

        try:
            output_path = self._services.fast_video_pipeline(
                prompt=request.prompt, image=None, size=request.size,
                duration_seconds=request.duration_s, on_progress=on_progress,
            )
            with self._lock:
                self._state.current_generation = GenerationComplete(
                    job_id=job_id, completed_at=datetime.utcnow(),
                    output_path=str(output_path),
                    duration_seconds=request.duration_s,
                )
        except Exception as e:
            with self._lock:
                self._state.current_generation = GenerationError(
                    job_id=job_id, failed_at=datetime.utcnow(),
                    error_message=str(e), error_kind=type(e).__name__,
                )


In [ ]:
# Drive the handler exactly the way the renderer does: start, then poll.
state = StateProvider()
runner = TaskRunner()
handler = AudioToVideoHandler(bundle, state, runner)

job_id = handler.start(A2VidRequest(audio_path="drums.wav", prompt="a drummer on a rooftop"))
print("job accepted:", job_id)

# The concurrency invariant holds: a second start while one is in flight is rejected.
try:
    handler.start(A2VidRequest(audio_path="other.wav"))
except GenerationAlreadyRunningError:
    print("second start rejected: GenerationAlreadyRunningError")

# Poll for state transitions, like the renderer polling /api/generation/progress.
seen = set()
while True:
    snap = handler.snapshot()
    desc = describe(snap)
    key = (desc["status"], desc.get("step"))
    if key not in seen:
        seen.add(key)
        print(desc)
    if desc["status"] in ("done", "error", "cancelled"):
        break
    time.sleep(0.005)

runner.wait_all()
print("recorded pipeline calls:", bundle.fast_video_pipeline.calls[-1])


The remaining LTX Desktop pieces are thin plumbing in the established style. Reference listings, not executed here (the route needs the app's dependency wiring, the fetch runs in the renderer):

```python
# From backend/_routes/audio_to_video.py (companion @ ch8-capstone)
@router.post("/api/audio-to-video", response_model=A2VidResponse)
async def start_a2vid(request: A2VidRequest, deps: BackendDeps = Depends(get_deps)):
    require_auth_token(deps.auth)
    job_id = deps.handlers.audio_to_video.start(request)
    return A2VidResponse(job_id=str(job_id))
```

```ts
// From frontend/api-client.ts (companion @ ch8-capstone)
export async function startAudioToVideo(req: A2VidRequest): Promise<{job_id: string}> {
  return backendFetch<A2VidResponse>('/api/audio-to-video', {
    method: 'POST',
    body: JSON.stringify(req),
  });
}
```

The full change is around 200 lines of Python and 80 lines of TypeScript across a dozen files, shipped as a code change rather than a plugin.

### ComfyUI: one duck-typed node file

The whole feature is one file a user drops into `custom_nodes/` and restarts. The cell below writes that file next to this notebook; it imports `ltx_pipelines`, so it executes only inside a ComfyUI install that has the LTX packages, which is why we write it rather than run it.

In [ ]:
%%writefile ltx_a2vid_node.py
# custom_nodes/ltx_a2vid_node.py
import torch
from ltx_pipelines.a2vid_two_stage import A2VidPipelineTwoStage
from ltx_pipelines.utils.types import OffloadMode
from ltx_core.quantization import QuantizationPolicy


# Module-level cache so repeated calls reuse the loaded model.
_PIPELINE_CACHE = None


def _get_pipeline(checkpoint_path, distilled_lora, upsampler_path, gemma_root):
    global _PIPELINE_CACHE
    if _PIPELINE_CACHE is None:
        _PIPELINE_CACHE = A2VidPipelineTwoStage(
            checkpoint_path=checkpoint_path,
            distilled_lora=distilled_lora,
            spatial_upsampler_path=upsampler_path,
            gemma_root=gemma_root,
            loras=[],
            quantization=QuantizationPolicy.fp8_cast(),
            offload_mode=OffloadMode.NONE,
        )
    return _PIPELINE_CACHE


class LTXAudioToVideo:
    @classmethod
    def INPUT_TYPES(cls):
        return {
            "required": {
                "audio_path": ("STRING", {"default": "", "multiline": False}),
                "prompt": ("STRING", {"default": "", "multiline": True}),
                "duration_seconds": ("FLOAT", {"default": 5.0, "min": 1.0, "max": 30.0, "step": 0.5}),
                "checkpoint_path": ("STRING", {"default": "/path/to/ltx-2.3-22b-dev.safetensors"}),
                "distilled_lora_path": ("STRING", {"default": "/path/to/ltx-2.3-22b-distilled-lora-384-1.1.safetensors"}),
                "upsampler_path": ("STRING", {"default": "/path/to/ltx-2.3-spatial-upscaler-x2-1.1.safetensors"}),
                "gemma_root": ("STRING", {"default": "/path/to/gemma-3-12b"}),
            }
        }

    RETURN_TYPES = ("STRING",)
    RETURN_NAMES = ("output_video_path",)
    FUNCTION = "generate"
    CATEGORY = "LTX/Audio-to-Video"

    def generate(self, audio_path, prompt, duration_seconds,
                 checkpoint_path, distilled_lora_path, upsampler_path, gemma_root):
        pipeline = _get_pipeline(
            checkpoint_path=checkpoint_path,
            distilled_lora=[(distilled_lora_path, 0.6)],
            upsampler_path=upsampler_path,
            gemma_root=gemma_root,
        )
        output = pipeline(
            audio_path=audio_path,
            prompt=prompt,
            duration_seconds=duration_seconds,
        )
        return (str(output),)


NODE_CLASS_MAPPINGS = {
    "LTXAudioToVideo": LTXAudioToVideo,
}

NODE_DISPLAY_NAME_MAPPINGS = {
    "LTXAudioToVideo": "LTX-2.3 Audio to Video",
}


Installation is the entire story: copy `ltx_a2vid_node.py` into ComfyUI's `custom_nodes/`, restart, and the node appears under LTX/Audio-to-Video. Wire a Load Audio node into `audio_path`, a string constant into `prompt`, and `output_video_path` into Save Video. About 50 lines, no manifest, no SDK, and also no auth, no integrated progress, and full process privileges: the price and the payoff of the lowest possible extension barrier.

### InvokeAI: one typed invocation

Between the two extremes: a single file in InvokeAI's `nodes/` directory, but with typed inputs that become UI automatically and progress events that feed the existing WebSocket channel. Written to a file for the same reason as the ComfyUI node.

In [ ]:
%%writefile ltx_a2vid_invocation.py
# nodes/ltx_a2vid_invocation.py
from invokeai.app.invocations.baseinvocation import (
    BaseInvocation,
    InputField,
    InvocationContext,
    invocation,
)
from invokeai.app.invocations.primitives import StringOutput

from ltx_pipelines.a2vid_two_stage import A2VidPipelineTwoStage
from ltx_pipelines.utils.types import OffloadMode
from ltx_core.quantization import QuantizationPolicy


@invocation(
    "ltx_audio_to_video",
    title="LTX-2.3 Audio to Video",
    tags=["ltx", "audio", "video"],
    category="generation",
)
class LTXAudioToVideoInvocation(BaseInvocation):
    """Generate a video synchronized to an input audio file using LTX-2.3."""

    audio_path: str = InputField(description="Path to input audio file (wav/mp3)")
    prompt: str = InputField(description="Text prompt for video content", default="")
    duration_seconds: float = InputField(
        default=5.0, ge=1.0, le=30.0,
        description="Output video duration in seconds",
    )
    checkpoint_path: str = InputField(description="Path to LTX-2.3 checkpoint")
    distilled_lora_path: str = InputField(description="Path to distilled LoRA")
    upsampler_path: str = InputField(description="Path to spatial upsampler")
    gemma_root: str = InputField(description="Path to Gemma 3 weights")

    def invoke(self, context: InvocationContext) -> StringOutput:
        pipeline = A2VidPipelineTwoStage(
            checkpoint_path=self.checkpoint_path,
            distilled_lora=[(self.distilled_lora_path, 0.6)],
            spatial_upsampler_path=self.upsampler_path,
            gemma_root=self.gemma_root,
            loras=[],
            quantization=QuantizationPolicy.fp8_cast(),
            offload_mode=OffloadMode.NONE,
        )

        def on_progress(step: int, total: int) -> None:
            context.events.emit_invocation_progress(
                queue_item_id=context.queue_item.item_id,
                node_id=self.id,
                progress=step / total,
                step=step,
            )

        output = pipeline(
            audio_path=self.audio_path,
            prompt=self.prompt,
            duration_seconds=self.duration_seconds,
            on_progress=on_progress,
        )
        return StringOutput(value=str(output))


### What the three implementations reveal

- LTX Desktop: about 280 lines across a dozen files, the deepest codebase knowledge, shipped as a fork or pull request. In exchange the feature inherits auth, progress channels, and every other pattern the app uses.
- ComfyUI: about 50 lines in one file, the least platform knowledge, installable by anyone. In exchange: no integrated progress, manual pipeline lifecycle, no sandboxing.
- InvokeAI: about 80 lines in one file. Typed inputs become UI, progress feeds the existing WebSocket channel, extension without touching core.

Same pipeline call at the center of all three. Choose the architecture that matches what you are optimizing for.

## Summary

The eight patterns, one line each: split UI and compute into two processes with an explicit boundary; make capability decisions once in one layer and publish them; derive both sides of every wire contract from one shared schema; treat localhost as a real trust boundary; treat model download, license, and storage as durable state with per-file progress; represent phase-specific state as discriminated unions; hide side effects behind Protocols and test with hand-written fakes; mutate state only under a lock you hold briefly, with heavy work in a runner.

The model is not the product. The system around the model is.